#### Import Library

In [68]:
from langchain_core.prompts import ChatPromptTemplate
from Project.Customer_Support_Ticket_Analyser.config import get_chat_model
from langchain_core.runnables import RunnableLambda

In [69]:
outputType = ['Category Name', 'Urgency', 'Sentiment', 'Small Summary']

#### Create the CHAT PROMPT for ChatPromptTemplate

In [70]:
classify_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI system that classifies customer support tickets.

Categories:
- Billing
- Technical Issue
- Account Access
- Feature Request
- Complaint

Return ONLY the category name, urgency, sentiment and small summary of the ticket.
"""
        ),
        ("human", "{ticket}")
    ]
)

In [71]:
print(classify_prompt.messages[0].prompt.template)


You are an AI system that classifies customer support tickets.

Categories:
- Billing
- Technical Issue
- Account Access
- Feature Request
- Complaint

Return ONLY the category name, urgency, sentiment and small summary of the ticket.



#### Invoke Chat Prompt Template

In [72]:
ticket_dict = {
    'ticket': 'My credit card was charged twice'
}

In [73]:
chat_template = classify_prompt.invoke(ticket_dict)

#### Invoke Chat to Classify System Message

In [74]:
chat = get_chat_model()

/Users/sahilnagpal/Library/Python/3.9/lib/python/site-packages/IPython/core/interactiveshell.py:3550: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [75]:
response = chat.invoke(chat_template)

In [76]:
response_list = response.content.split(", ")

#### Parse JSON Output

In [83]:
def parseResponseJSON(outputType, response_list):
    cleaned_list = [item.replace('\n', ' ') for item in response_list]
    return dict(zip(outputType, cleaned_list))

#### Create Chain of Response

In [84]:
chain = classify_prompt | chat | RunnableLambda(lambda x: parseResponseJSON(response_list=response_list, outputType=outputType))
result = chain.invoke(ticket_dict)

In [85]:
result

{'Category Name': "Category: Billing Urgency: High Sentiment: Negative Summary: The customer's credit card has been double charged."}